# Subvolume Dense Reconstruction Set

**Purpose:** Processes a densely-reconstructed EM subvolume to extract per-cell segment IDs.  
Bounding box x = 14,125:17,875 y = 667:2,667 (with 16nm seg scaling)  
bounding box x: 907-1141 and z: 103-325
isotropic resolution: 256,256,270

so box in microns = 60 X 60 microns

**Input** Exports from VAST (directly and via Matlab scripts) for a segmentation layer mask on the isotropic EM volume (to capture soma segments and register them with reconstructed neurons in the dataset and then obtain any missing reconstructions). 

**Output** Dataframe (.csv) of all of the cells (by main/anchor segment ID) within the bounding box constraining the densely reconstructed subvolume.



In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from copy import deepcopy
from tqdm import tqdm
import scipy.io
from PIL import Image
import cloudvolume

from efish_em import AnalysisCode as efish #only works after the efish_em package has been installed as per README

/Users/kperks/opt/anaconda3/envs/efish_em_tier3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
folder_path = DATA_ROOT / 'data_VAST/volume_subsample_sg-mg-out_ratio'
png_path = DATA_ROOT / 'data_VAST/iso_thumbnails_mSEM/16nm_EM_png_stack_5x5'

df_type = pd.read_csv(DATA_ROOT / 'data_processed_published/df_type_auto_typed.csv')

## Get list of all reconstructions with anchor_seg within bounding box

bounding box x: 907-1141 and z: 103-325
isotropic resolution: 256,256,270

so box in microns = 60 X 60 microns

In [20]:
efish_cloudvolume = cloudvolume.CloudVolume('gs://efish-public/roi450um_seg32fb16fb_220930/|neuroglancer-precomputed:', progress=False)

## find missing z sections

In [14]:
# Detect which z sections are present in the isotropic EM stack.
# Missing slices are empty PNGs (sum == 0) and must be excluded from the z-mapping.

files = sorted([file for file in png_path.iterdir() if file.is_file()])

img_list_sum = []
with tqdm(total=len(files)) as pbar:
    for f in files:
        pbar.update(1)
        img = Image.open(f)
        # Convert the image to a NumPy array
        img_list_sum.append(np.array(img).sum())

missing_slices = np.where(np.array(img_list_sum)==0)[0]

slices_used = [x for x in range(0, 3535) if x not in missing_slices]

actual_iso_z_map = np.array(slices_used[::8])+3


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3536/3536 [00:00<00:00, 6116.70it/s]


## import actual coords

isotropic soma segments within bounding box x: 907-1141 and z: 103-325
No inclusion/exclusion planes accounted for yet

In [22]:
# Load the isotropic sample coordinates from the VAST subvolume export.

mat = scipy.io.loadmat(folder_path / 'actual_coords_isotropic.mat')
data = mat['actual_coords']  # Access the data from the loaded .mat file

# Create a pandas DataFrame from the data.
df = pd.DataFrame(data, columns=('x','y','z'), index=range(1,len(data)+1))


        x       y      z
1     0.0     0.0    0.0
2  1024.0   993.0  102.0
3  1004.0  1022.0   98.0
4  1050.0  1002.0  101.0
5  1037.0  1019.0   94.0
(380, 3)


In [23]:
df = df.dropna()
df = df.iloc[1:] #remove background

In [25]:
# convert columns of df from isotropic scaling to 16nm segmentation scaling
df.loc[:,'x_adj'] = df['x']*256/16
df.loc[:,'y_adj'] = df['y']*256/16
# df.loc[:,'z'] = df['z']*270/30

In [26]:
df.loc[:,'z_adj'] = [actual_iso_z_map[int(z)] for z in df['z'].values]

In [27]:
with tqdm(total=len(df)) as pbar:
    for i,r in df.iterrows():
        pbar.update(1)
        
        df.loc[i,'segment']= int(efish_cloudvolume[[int(r['x_adj']), int(r['y_adj']), int(r['z_adj'])]][0][0][0][0])
        

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 371/371 [02:30<00:00,  2.47it/s]


In [29]:
df_segments=deepcopy(df)

In [228]:
# df.to_csv(folder_path / 'df_segments.csv') # (folder_path was set in the setup cell to DATA_ROOT / 'data_VAST/volume_subsample_sg-mg-out_ratio')

## load VAST segment location data from Matlab

In [ ]:
# df_segments = pd.read_csv(folder_path / 'df_segments.csv') # (folder_path was set in the setup cell to DATA_ROOT / 'data_VAST/volume_subsample_sg-mg-out_ratio')

In [31]:
df_segments['segment']= df_segments['segment'].astype(int)

In [34]:
# base_segments maps each eCREST cell_graph JSON file to the set of base segment IDs it contains.
# dirpath points at the published reconstruction folder (set in the setup cell).
    
base_segments = efish.get_base_segments_dict(dirpath)


In [35]:
# for each row in df_segments
for i,r in df_segments.iterrows():
    # print(r)

    segid_to_find = int(r['segment'])
    # print(segid_to_find)

    cell_id = list(set([k.split('_')[2] for k,v in base_segments.items() if str(segid_to_find) in v]))
    # print(cell_id)
    if cell_id == []:
        cell_id = np.nan
        cell_type = 'none'
    elif cell_id != []:
        if len(cell_id)>1:
            print(f'{len(cell_id)} cells match: {cell_id}')
        if len(cell_id)==1:
            cell_id = cell_id[0]
            # print(cell_id)
            # print(df_type[df_type['id'].isin([cell_id])]['cell_type'])
            cell_type = df_type[df_type['id'].isin([int(cell_id)])]['cell_type'].values[0]
        
    # print(i,segid_to_find,cell_id)

    # enter cell_id in new column (empty or NaN if none)
    df_segments.loc[i,'cell_id'] = cell_id
    df_segments.loc[i,'cell_type']=cell_type



In [36]:
mask = (df_segments['segment']==0)
df_segments[mask]

,x,y,z,x_adj,y_adj,z_adj,segment,cell_id,cell_type
76,939.0,1014.0,144.0,15024.0,16224.0,1225,0,NaN,none
114,1028.0,1211.0,174.0,16448.0,19376.0,1479,0,NaN,none
138,916.0,1026.0,174.0,14656.0,16416.0,1479,0,NaN,none
281,1121.0,1035.0,253.0,17936.0,16560.0,2117,0,NaN,none
296,1031.0,1177.0,224.0,16496.0,18832.0,1882,0,NaN,none
339,942.0,1098.0,198.0,15072.0,17568.0,1674,0,NaN,none


In [382]:
df_segments[df_segments['cell_type'].isin(['sg1'])]['cell_id'].values

array(['128800360', '128816291', '128849661', '218170254', '220398612',
       '213528973', '219284058', '216994785', '216949671', '217026667',
       '215820918', '214627855', '214736845', '215835078', '214707822',
       '218109807', '215774633', '217027723', '215883144', '302777004',
       '304015064', '301677416', '301693439', '300565422', '300596446',
       '301756099', '303953883', '563945132', '473493287', '559349295',
       '558189118', '473461904', '558189299', '474621530', '475766527',
       '473460724', '475765678', '475812081', '475810650', '391083807',
       '476894379', '474542263', '473444334', '386363638', '387634327',
       '474573138', '474588143', '389845032', '389798775', '387585036',
       '387554847', '386393755', '385357762', '388730139', '388667843',
       '389889717', '387647235', '392102325', '386455093', '386424384',
       '386470356', '386501395', '389858244', '300489438', '386501395',
       '301648806', '300489438', '300582622'], dtype=object)

## Account for any missing segments manually

In [37]:
mask = (df_segments['cell_type']=='none') & (df_segments['segment']==0)
df_segments[mask]#['segment'].values
# df_segments[mask][['x_adj','y_adj','z_adj']].values

,x,y,z,x_adj,y_adj,z_adj,segment,cell_id,cell_type
76,939.0,1014.0,144.0,15024.0,16224.0,1225,0,NaN,none
114,1028.0,1211.0,174.0,16448.0,19376.0,1479,0,NaN,none
138,916.0,1026.0,174.0,14656.0,16416.0,1479,0,NaN,none
281,1121.0,1035.0,253.0,17936.0,16560.0,2117,0,NaN,none
296,1031.0,1177.0,224.0,16496.0,18832.0,1882,0,NaN,none
339,942.0,1098.0,198.0,15072.0,17568.0,1674,0,NaN,none


In [38]:
mask_inds = df_segments[mask].index

In [39]:
maskseg_inds = [214645195,308580905,300503321,386488078,392163961,303954925]

In [40]:
for sid in maskseg_inds:
    print(list(set([k.split('_')[2] for k,v in base_segments.items() if str(sid) in v])))

['213501262']
['307418797']
['300503092']
['386488569']
['306291045']
['303954781']


In [41]:
for i,s in zip(mask_inds,maskseg_inds):
    df_segments.loc[i,'segment'] = s
    cell_id = list(set([k.split('_')[2] for k,v in base_segments.items() if str(s) in v]))

    if cell_id == []:
        cell_id = np.nan
        cell_type = 'none'
    elif cell_id != []:
        if len(cell_id)>1:
            print(f'{len(cell_id)} cells match: {cell_id}')
        if len(cell_id)==1:
            cell_id = cell_id[0]
            cell_type = df_type[df_type['id'].isin([int(cell_id)])]['cell_type'].values[0]

    # enter cell_id in new column (empty or NaN if none)
    df_segments.loc[i,'cell_id'] = cell_id
    df_segments.loc[i,'cell_type']=cell_type


In [42]:
df_segments.loc[mask_inds,:]

,x,y,z,x_adj,y_adj,z_adj,segment,cell_id,cell_type
76,939.0,1014.0,144.0,15024.0,16224.0,1225,214645195,213501262,mg1
114,1028.0,1211.0,174.0,16448.0,19376.0,1479,308580905,307418797,lf
138,916.0,1026.0,174.0,14656.0,16416.0,1479,300503321,300503092,mg2
281,1121.0,1035.0,253.0,17936.0,16560.0,2117,386488078,386488569,sg2
296,1031.0,1177.0,224.0,16496.0,18832.0,1882,392163961,306291045,h
339,942.0,1098.0,198.0,15072.0,17568.0,1674,303954925,303954781,sg1


In [49]:
missing_cell_1 = {#'Unnamed: 0': 373,
    'x':1027,
    'y':1176,
    'z':234,
    'x_adj':16436,
    'y_adj':18812, 
    'z_adj':1902,
    'segment':392164712,
    'cell_id':392164931,
    'cell_type':df_type[df_type['id'].isin([392164931])]['cell_type'].values[0]
               }

missing_cell_2 = {#'Unnamed: 0': 374,
    'x':905,
    'y':1287,
    'z':2182,
    'x_adj':14494,
    'y_adj':20596, 
    'z_adj':2355,
    'segment':482558928,
    'cell_id':483687142,
    'cell_type':df_type[df_type['id'].isin([483687142])]['cell_type'].values[0]
               }



In [ ]:
missing_df = pd.concat([pd.DataFrame([missing_cell_1]), pd.DataFrame([missing_cell_2])], ignore_index=True)

df_segments = pd.concat([df_segments,missing_df], ignore_index=True)

df_segments['segment'] = df_segments['segment'].astype(int)

In [60]:
df_segments.to_csv(folder_path / 'df_segments_assigned.csv', index=False)